# 01 — Market handles

`tq.MarketEnvironment`: rate curves, discrete dividends, vol surfaces. Q/A: `docs/chapters/rates.tex` (Curves), `equity.tex`.

In [ ]:
import os, sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# examples/ holds market_data.py (synthetic market)
REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "examples"))

import numpy as np
import matplotlib.pyplot as plt
import tensorquant as tq
from market_data import EVALUATION_DATE, build_market_environment

## Market
Synthetic market from `examples/market_data.py`.

In [ ]:
tq.Settings.evaluation_date = EVALUATION_DATE
mkt = build_market_environment(EVALUATION_DATE)

for k in sorted(mkt._market):
    print(f"{k:<38s} {type(mkt._market[k]).__name__}")

## Rate curves

Key relations: $P(0,T)=e^{-r(T)T}$, $\quad F(0;T_1,T_2)=\frac{1}{\tau}\left(\frac{P(0,T_1)}{P(0,T_2)}-1\right)$, $\quad f(0,T)=r(T)+T\,r'(T)$.

In [ ]:
ir_tickers = [k.split(":")[2] for k in sorted(mkt._market)
              if k.startswith("IR:EUR:") and k.endswith(":SPOT")]

t = np.linspace(0.25, 10.0, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ticker in ir_tickers:
    c = mkt.get_ir_curve(tq.Currency.EUR, ticker=ticker)
    df = np.array([float(c.discount(x)) for x in t])
    axes[0].plot(t, -np.log(df) / t * 100, label=ticker)
    axes[1].plot(t, df, label=ticker)
    axes[2].plot(t, [float(c.inst_fwd(x)) * 100 for x in t], label=ticker)
for ax, title in zip(axes, ["Zero rate cc (%)", "Discount factor", "Inst. forward (%)"]):
    ax.set_title(title); ax.set_xlabel("T (Y)"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()

## Discrete dividends

In [ ]:
for k in sorted(mkt._market):
    if k.startswith("EQ:") and k.endswith(":DIV"):
        ccy, ticker = tq.Currency[k.split(":")[1]], k.split(":")[2]
        spot = float(mkt.get_eq_spot(ticker, ccy=ccy))
        div = mkt.get_eq_dividends(ticker, ccy=ccy)
        print(f"{ticker}: spot={spot:.2f}")
        for ex, amt in zip(div.ex_dates, div.amounts):
            print(f"   {ex}  {amt:8.4f}  ({amt/spot:.2%})")

## Volatility smiles

In [ ]:
for k in sorted(mkt._market):
    if k.startswith("EQ:") and k.endswith(":VOL"):
        ccy, ticker = tq.Currency[k.split(":")[1]], k.split(":")[2]
        vs = mkt.get_eq_vol_surface(ticker, ccy=ccy)
        spot = float(mkt.get_eq_spot(ticker, ccy=ccy))
        K = np.array(vs.strike)
        plt.figure(figsize=(6, 4))
        for T in vs.maturity:
            plt.plot(K / spot, [float(vs.volatility(strike=x, tenor=T)) * 100 for x in K], "o-", ms=3, label=f"T={T:.2f}")
        plt.title(f"{ticker} smile"); plt.xlabel("K / spot"); plt.ylabel("IV (%)")
        plt.grid(alpha=0.3); plt.legend(fontsize=7, ncol=2)

## Exercises
1. Check numerically $f(0,T) \approx r(T) + T\,r'(T)$ with finite differences.
2. Compute the equity forward $F(0,T)=\big(S_0-\mathrm{PV(div)}\big)/P(0,T)$ for one ticker.